### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "SMOTE" / "gridsearched_predictors" / "RandomForest_thres0.5_2026-05-07" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.8,,,,,
1,0,cf_1,1,,,,,,18.9007,,,0.878,2,False,0.2426,0.3046
2,0,cf_2,,,,,,,15.3541,,,1.0,1,False,0.2426,0.2499
3,0,cf_3,3,,,,,,18.3402,,,0.8972,2,False,0.2426,0.2945
4,0,cf_4,,,,6,,,17.8386,,,0.9145,2,False,0.2426,0.4916
5,0,cf_5,,,,,1,,17.5923,,,0.923,2,False,0.2426,0.2292
6,0,cf_6,,,6,,,,17.445,,,0.928,2,False,0.2426,0.1764
7,0,cf_7,,,,,,,16.9362,5,,0.9456,2,False,0.2426,0.1921
8,0,cf_8,,,,7,,,18.9745,7,,0.8754,3,False,0.2426,0.1617
9,0,cf_9,,1,,,,,18.9659,7,,0.8757,3,False,0.2426,0.1917


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.8,,,,,
15,0,cf_1,1,,,,,,18.9007,,,0.878,2,False,0.2426,0.3046
1,1,xin,3,4,1,2,3,0,22.3757,0,3.03,,,,,
9,1,cf_4,,,,3,1,,22.3757,,,0.8796,3,True,0.4963,0.1538
10,1,cf_6,,,,,1,,22.3757,1,,0.8796,3,True,0.4963,0.1855
11,1,cf_8,2,,,,1,,22.3757,,,0.8796,3,True,0.4963,0.1084
12,1,cf_9,,,2,,1,,22.3757,,,0.8796,3,True,0.4963,0.1956
2,2,xin,5,3,1,1,4,0,22.694,7,3.41,,,,,
16,2,cf_1,,,,,,,22.6835,,,0.9471,1,False,0.3535,0.3535
3,3,xin,3,3,6,6,2,0,24.3418,1,9.59,,,,,


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.8,,,,,
9,0,cf_1,1,,,,,,18.9007,,,0.878,2,False,0.2426,0.3046
1,1,xin,3,4,1,2,3,0,22.3757,0,3.03,,,,,
10,1,cf_6,,,,,1,,22.3757,1,,0.8796,3,True,0.4963,0.1855
2,2,xin,5,3,1,1,4,0,22.694,7,3.41,,,,,
11,2,cf_1,,,,,,,22.6835,,,0.9471,1,False,0.3535,0.3535
3,3,xin,3,3,6,6,2,0,24.3418,1,9.59,,,,,
12,3,cf_1,,,,,,,24.3375,,,0.8751,1,False,0.4107,0.4092
4,4,xin,5,4,2,7,1,0,21.2585,3,4.63,,,,,
13,4,cf_1,,,,,,,21.2585,,,0.9868,1,False,0.0215,0.0215


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.3046
Valid: False
Changes:
  etfruit: 4 → 1
  bmi: 18.9866 → 18.9007


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_4 ---
Predicted risk: 0.1538
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_6 ---
Predicted risk: 0.1855
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_8 ---
Predicted risk: 0.1084
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1

--- cf_9 ---
Predicted risk: 0.1956
Valid: True
Changes:
  cgtsmok: 1 → 2
  slprl: 3 → 1


=== Query index '2' ===
Task / Target: hltprhc
Query instance index: 2

Original instance:
  et